# Object detection using PyTorch

Popular object detection models:
- **YOLO** (You Only Look Once) - Fast and accurate.
- **SSD** (Single Shot MultiBox Detector) - Real-time object detection.
- **Faster R-CNN** (Region-based Convolutional Neural Networks) - High accuracy.

import the necessary libraries

In [1]:
import os
import shutil

import kagglehub
import pandas as pd
import torch
import torchvision
from PIL import Image
from torch.utils.data import Dataset
from torchvision import transforms
from tqdm import tqdm

check the versions of PyTorch and Torchvision

In [2]:
print(f'PyTorch Version: {torch.__version__}')
print(f'Torchvision Version: {torchvision.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')

PyTorch Version: 2.13.0+cpu
Torchvision Version: 0.28.0+cpu
CUDA Available: False


download the dataset from Kaggle (Global Wheat Head Dataset 2021) and prepare it for YOLO format

In [3]:
path = kagglehub.dataset_download("vbookshelf/global-wheat-head-dataset-2021",
                                  output_dir='original_dataset',
                                  force_download=True)

print("Path to dataset files:", path)

In [4]:
csv_path = './original_dataset/competition_train.csv'
image_dir = './original_dataset/images'

create a custom dataset class for YOLO format

In [5]:
class WheatDatasetYOLO(Dataset):
    def __init__(self, annotations_file, img_dir, transform=None):
        self.annotations = pd.read_csv(annotations_file)
        self.img_dir = img_dir
        self.transform = transform
        self.image_ids = self.annotations['image_name'].unique()

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        img_path = os.path.join(self.img_dir, image_id)
        image = Image.open(img_path).convert("RGB")
        width, height = image.size

        # Getting all boxes for this image
        records = self.annotations[self.annotations['image_name'] == image_id]
        boxes = []
        for _, row in records.iterrows():
            box_str = row['BoxesString']
            if pd.isna(box_str) or box_str.strip().lower() == 'no_box':
                continue
            for b in box_str.split(';'):
                x1, y1, x2, y2 = map(float, b.strip().split())
                xc = (x1 + x2) / 2 / width
                yc = (y1 + y2) / 2 / height
                w = (x2 - x1) / width
                h = (y2 - y1) / height
                boxes.append([0, xc, yc, w, h])  # class 0 for wheat

        boxes = torch.tensor(boxes, dtype=torch.float32)

        if self.transform:
            image = self.transform(image)

        return image, boxes

initialize the dataset and check its length and a sample

In [6]:
transform = transforms.Compose([transforms.ToTensor()])
dataset = WheatDatasetYOLO(
    annotations_file=csv_path,
    img_dir=image_dir,
    transform=transform
)

print("Dataset length:", len(dataset))
img, labels = dataset[0]
print("Image shape:", img.shape)
print("YOLO segments:", labels)

Dataset length: 3655
Image shape: torch.Size([3, 1024, 1024])
YOLO labels: tensor([[0.0000, 0.1265, 0.7109, 0.0596, 0.0703],
        [0.0000, 0.6533, 0.0693, 0.0547, 0.0859],
        [0.0000, 0.9507, 0.9756, 0.0752, 0.0410],
        [0.0000, 0.4102, 0.8418, 0.0840, 0.0547],
        [0.0000, 0.6592, 0.7979, 0.0410, 0.0391],
        [0.0000, 0.2690, 0.7954, 0.0674, 0.0908],
        [0.0000, 0.2759, 0.9160, 0.1025, 0.0527],
        [0.0000, 0.3618, 0.7705, 0.0967, 0.0703],
        [0.0000, 0.6802, 0.5312, 0.0869, 0.0430],
        [0.0000, 0.0444, 0.0425, 0.0889, 0.0576],
        [0.0000, 0.8115, 0.0811, 0.0957, 0.0547],
        [0.0000, 0.8052, 0.7271, 0.0322, 0.0596],
        [0.0000, 0.0322, 0.9102, 0.0645, 0.0332],
        [0.0000, 0.9292, 0.2402, 0.0732, 0.0527],
        [0.0000, 0.7959, 0.4692, 0.0723, 0.0615],
        [0.0000, 0.3652, 0.9136, 0.0469, 0.0479],
        [0.0000, 0.1309, 0.8394, 0.0586, 0.0381],
        [0.0000, 0.8091, 0.8560, 0.0811, 0.0557],
        [0.0000, 0.8096, 

create directories for YOLO formatted dataset and split into train and validation sets

In [7]:
output_base = 'data_source/wheat'
os.makedirs(output_base, exist_ok=True)

# Subfolders
for sub in ['images/train', 'segments/train', 'images/val', 'segments/val']:
    os.makedirs(os.path.join(output_base, sub), exist_ok=True)

process the dataset to convert annotations to YOLO format and copy images to the respective directories

In [8]:
# Reading annotations
df = pd.read_csv(csv_path)
all_image_ids = df['image_name'].unique()
dataset_max_using_pct = 0.1
subset_size = int(dataset_max_using_pct * len(all_image_ids))
image_ids = all_image_ids[:subset_size]  # take top 10%

val_split = 0.1
val_count = int(len(image_ids) * val_split)
val_ids = set(image_ids[:val_count])

# Converting box to YOLO format
def convert_box(img_width, img_height, x1, y1, x2, y2):
    x_center = ((x1 + x2) / 2) / img_width
    y_center = ((y1 + y2) / 2) / img_height
    width = (x2 - x1) / img_width
    height = (y2 - y1) / img_height
    return [0, x_center, y_center, width, height]

# Generating YOLO segments and copying files
for img_id in tqdm(image_ids, desc=f"Converting {int(dataset_max_using_pct * 100)}% dataset"):
    label_rows = df[df['image_name'] == img_id]
    img_path = os.path.join(image_dir, img_id)
    if not os.path.exists(img_path): continue

    try:
        with Image.open(img_path) as im:
            w, h = im.size
    except:
        continue

    yolo_lines = []
    for _, row in label_rows.iterrows():
        if pd.isna(row['BoxesString']) or row['BoxesString'].strip().lower() == 'no_box':
            continue
        for box in row['BoxesString'].split(';'):
            x1, y1, x2, y2 = map(float, box.strip().split())
            yolo_box = convert_box(w, h, x1, y1, x2, y2)
            yolo_lines.append(' '.join(map(str, yolo_box)))

    subset = 'val' if img_id in val_ids else 'train'
    shutil.copy(img_path, f"{output_base}/images/{subset}/{img_id}")

    # Writing label
    label_path = f"{output_base}/segments/{subset}/{img_id.replace('.jpg', '.txt').replace('.png', '.txt')}"
    with open(label_path, 'w') as f:
        f.write('\n'.join(yolo_lines))

Converting 10% dataset: 100%|██████████| 365/365 [00:04<00:00, 90.27it/s] 
